In [ ]:
import json
import os
import re
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

# Set fonts.
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 300

# Data directory.
base_dir = "paper/Analysis/Needle_Position/results"

# Position order, from start to end.
positions = ['position_start', 'position_early_mid', 'position_late_mid', 'position_end']
position_labels = ['Start', 'Early Middle', 'Late Middle', 'End']

# Model list.
models = ['gpt-oss-120b-medium']
model_labels = ['GPT-oss-120b']

def load_position_data(base_dir, position, model):
    """Load safety rate data from a specific position and model"""
    # data[level][context_length] = safe_rate
    data = defaultdict(dict)
    
    model_dir = os.path.join(base_dir, position, model)
    if not os.path.exists(model_dir):
        print(f"Directory not found: {model_dir}")
        return data
    
    for filename in os.listdir(model_dir):
        if filename.startswith('eval_level_'):
            # Parse filename: eval_level_X_Yk_results_google-gemini-2.0-flash-lite-001.json
            match = re.match(r'eval_level_(\d+)_(\d+)k_results_google-gemini-2.0-flash-lite-001.json', filename)
            if match:
                level = int(match.group(1))
                context_length = int(match.group(2))
                
                filepath = os.path.join(model_dir, filename)
                try:
                    with open(filepath, 'r', encoding='utf-8') as f:
                        result = json.load(f)
                        stats = result.get('statistics', {})
                        total = stats.get('total', 0)
                        score_dist = stats.get('score_distribution', {})
                        # score=5 as unsafe
                        score_5_count = score_dist.get('5', 0)
                        
                        if total > 0:
                            safe_rate = (total - score_5_count) / total * 100
                            data[level][context_length] = safe_rate
                        else:
                            data[level][context_length] = 0.0
                except Exception as e:
                    print(f"Error reading {filepath}: {e}")
    
    return data

# Load all data.
# all_data[model][position][level][context_length] = safe_rate
all_data = {}
for model in models:
    all_data[model] = {}
    for position in positions:
        all_data[model][position] = load_position_data(base_dir, position, model)

# Get all context lengths and levels.
all_context_lengths = set()
all_levels = set()
for model in models:
    for position in positions:
        for level, ctx_data in all_data[model][position].items():
            all_levels.add(level)
            all_context_lengths.update(ctx_data.keys())

all_context_lengths = sorted(list(all_context_lengths))
all_levels = sorted(list(all_levels))

print(f"Context Lengths: {all_context_lengths}")
print(f"Levels: {all_levels}")


In [ ]:
# Lost in the Middle-style figure.
# X-axis: Position from start to end.
# Y-axis: Safety rate.
# Different line colors represent different context lengths.

# Select context lengths to display.
display_ctx_lengths = [0, 1, 4, 8, 16, 32, 64]

# Exclude level 0 and average only levels 1, 2, and 3.
levels_to_avg = [l for l in all_levels if l != 0]
print(f"Averaging over levels: {levels_to_avg}")

# Color mapping from light to dark.
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(display_ctx_lengths)))

# Single plot.
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

for model_idx, (model, model_label) in enumerate(zip(models, model_labels)):
    
    # Calculate the average safety rate across levels 1, 2, and 3 for each context length.
    for ctx_idx, ctx_len in enumerate(display_ctx_lengths):
        position_rates = []
        
        for position in positions:
            # Calculate the average safety rate across levels 1, 2, and 3 at this position, excluding level 0.
            rates = []
            for level in levels_to_avg:
                if ctx_len in all_data[model][position].get(level, {}):
                    rates.append(all_data[model][position][level][ctx_len])
            
            if rates:
                position_rates.append(np.mean(rates))
            else:
                position_rates.append(np.nan)
        
        # Draw lines.
        x_pos = np.arange(len(positions))
        ax.plot(x_pos, position_rates, 
                color=colors[ctx_idx], 
                marker='o', 
                markersize=12, 
                linewidth=3,
                label=f'{ctx_len}k',
                markerfacecolor=colors[ctx_idx], 
                markeredgecolor='white', 
                markeredgewidth=2)
    
    ax.set_xlabel('Needle Position', fontsize=20)
    ax.set_ylabel('Avg Safety Ratio (%)', fontsize=20)
    # ax.set_title(f'{model_label}', fontsize=22, fontweight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(position_labels, fontsize=18)
    ax.set_ylim(30, 102)  # Adjust the Y-axis range to include the 64k data.
    ax.set_yticks([50, 60, 70, 80, 90, 100])
    ax.tick_params(axis='y', labelsize=16)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(title='Context Length', loc='lower right', fontsize=14, title_fontsize=15,
              ncol=2, framealpha=0.9)  # Use a two-column legend for a more compact layout.

plt.tight_layout()
plt.savefig('needle_position_lost_in_middle.pdf', dpi=300, bbox_inches='tight')
plt.show()
